In [2]:
import pandas as pd

def merge_csv_files(csv_path1, csv_path2, output_path):
    # Read both CSV files
    df1 = pd.read_csv(csv_path1)
    df2 = pd.read_csv(csv_path2)

    # Print row counts of the input files
    print(f"📄 Rows in {csv_path1}: {len(df1)}")
    print(f"📄 Rows in {csv_path2}: {len(df2)}")

    # Concatenate the dataframes
    merged_df = pd.concat([df1, df2], ignore_index=True)

    # Print row count of the output file
    print(f"✅ Total rows in merged file: {len(merged_df)}")

    # Save the merged dataframe to output CSV
    merged_df.to_csv(output_path, index=False)
    print(f"📁 Merged file saved to: {output_path}")

# Example usage:
merge_csv_files(
    csv_path1="Alabama/final_1_of_2.csv",
    csv_path2="Alabama/final_2_of_2.csv",
    output_path="Alabama/final.csv"
)


📄 Rows in Alabama/final_1_of_2.csv: 7733
📄 Rows in Alabama/final_2_of_2.csv: 16213
✅ Total rows in merged file: 23946
📁 Merged file saved to: Alabama/final.csv


In [3]:
import os
import json
import numpy as np
import pandas as pd
from shapely.geometry import Polygon, mapping
import fiona

def save_polygon_shapefile_from_mask(csv_path, output_dir, reference_crs="EPSG:26916"):
    """
    Generates a shapefile using polygon coordinates from the 'mask_points_gis' column in the CSV.
    """

    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    shapefile_path = os.path.join(output_dir, "polygon_shapes.shp")

    # Load CSV
    df = pd.read_csv(csv_path)

    # Define schema for attributes in shapefile
    schema = {
        'geometry': 'Polygon',
        'properties': {
            'latitude': 'float',
            'longitude': 'float',
            'class': 'int',
            'confidence': 'float',
            'tile_name': 'str:50',
            'area_geo': 'float'
        }
    }

    # Open shapefile writer
    with fiona.open(
        shapefile_path,
        mode='w',
        driver='ESRI Shapefile',
        crs=reference_crs,
        schema=schema
    ) as shp:
        for idx, row in df.iterrows():
            try:
                # Parse mask_points_gis column into a list of coordinate pairs
                mask_str = row.get('mask_points_gis', '[]')
                if not mask_str.strip():
                    continue

                coords = json.loads(mask_str.replace('(', '[').replace(')', ']'))

                # Ensure the coordinates form a closed polygon
                if coords[0] != coords[-1]:
                    coords.append(coords[0])

                if len(coords) < 3:
                    continue  # Not enough points for a valid polygon

                # Create polygon
                poly = Polygon(coords)

                if not poly.is_valid or poly.is_empty:
                    continue

                # Safely get and encode tile name
                tile = str(row.get('tile_name', '')).encode('ascii', 'ignore').decode()[:50]

                # Write the polygon feature
                shp.write({
                    'geometry': mapping(poly),
                    'properties': {
                        'latitude': float(row.get('latitude', 0)),
                        'longitude': float(row.get('longitude', 0)),
                        'class': int(row.get('class', 0)),
                        'confidence': float(row.get('confidence', 0)),
                        'tile_name': tile,
                        'area_geo': float(row.get('area_geo', 0))
                    }
                })

            except Exception as e:
                print(f"⚠️ Skipped row {idx} due to error: {e}")

    print(f"\n✅ Shapefile successfully written to: {shapefile_path}")


save_polygon_shapefile_from_mask("Alabama/final.csv", "Alabama/shapefile_output")


✅ Shapefile successfully written to: Alabama/shapefile_output\polygon_shapes.shp


In [4]:
import geopandas as gpd
from shapely.ops import unary_union
from shapely.strtree import STRtree
import os

def merge_overlapping_polygons_with_largest_area_attr(shapefile_path, output_path):
    # Load shapefile
    gdf = gpd.read_file(shapefile_path)

    # Ensure CRS is projected (not geographic) for accurate area computation
    if gdf.crs.is_geographic:
        print("🔁 Reprojecting to a projected CRS for accurate area calculation...")
        gdf = gdf.to_crs(epsg=26916)  # UTM Zone 16N as default for EPSG:26916

    # Compute area_gis and store in new column
    gdf["area_gis"] = gdf.geometry.area

    # Prepare for spatial merging
    geometries = list(gdf.geometry)
    str_tree = STRtree(geometries)
    visited = set()
    merged_geoms = []
    retained_rows = []

    for idx, geom in enumerate(geometries):
        if idx in visited:
            continue

        # Find all overlapping geometries
        overlaps = [i for i in str_tree.query(geom) if gdf.geometry[i].intersects(geom)]
        overlaps = list(set(overlaps) - visited)

        if not overlaps:
            continue

        # Merge all overlapping geometries
        merged_geom = unary_union(gdf.geometry.iloc[overlaps])

        # Retain the row with largest area_gis
        max_idx = gdf.iloc[overlaps]["area_gis"].idxmax()
        retained_row = gdf.loc[max_idx].copy()
        retained_row["geometry"] = merged_geom

        merged_geoms.append(merged_geom)
        retained_rows.append(retained_row)
        visited.update(overlaps)

    # Create new GeoDataFrame with merged polygons
    merged_gdf = gpd.GeoDataFrame(retained_rows, crs=gdf.crs)

    # Save to output
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    merged_gdf.to_file(output_path)

    print(f"\n✅ Merged shapefile saved to: {output_path}")

# Example usage:
merge_overlapping_polygons_with_largest_area_attr(
    shapefile_path="Alabama/shapefile_output/polygon_shapes.shp",
    output_path="Alabama/shapefile_output/merged_polygons.shp"
)


✅ Merged shapefile saved to: Alabama/shapefile_output/merged_polygons.shp


In [5]:
import geopandas as gpd
import numpy as np
import os

def process_shapefile_with_bbox_dimensions(shapefile_path, output_path):
    # Load shapefile
    gdf = gpd.read_file(shapefile_path)

    # Reproject to UTM (feet) if not already projected
    if gdf.crs.is_geographic:
        print("🔁 Reprojecting to projected CRS for accurate measurement...")
        gdf = gdf.to_crs(epsg=26916)  # UTM Zone 16N (meters)
    
    # Convert from meters to feet after area calculation
    sq_m_to_sq_ft = 10.7639
    m_to_ft = 3.28084

    # Calculate fresh area in sq ft
    gdf["area_gis"] = gdf.geometry.area * sq_m_to_sq_ft

    # Compute length and width from minimum rotated rectangle
    lengths = []
    widths = []

    for geom in gdf.geometry:
        # Get the minimum rotated rectangle
        min_rect = geom.minimum_rotated_rectangle

        # Get the rectangle's 4 corners
        coords = list(min_rect.exterior.coords)

        # Compute side lengths
        d1 = np.linalg.norm(np.array(coords[0]) - np.array(coords[1]))
        d2 = np.linalg.norm(np.array(coords[1]) - np.array(coords[2]))

        # Convert to feet
        d1_ft = d1 * m_to_ft
        d2_ft = d2 * m_to_ft

        # Assign length = max, width = min
        length_ft = max(d1_ft, d2_ft)
        width_ft = min(d1_ft, d2_ft)

        lengths.append(length_ft)
        widths.append(width_ft)

    gdf["length"] = lengths
    gdf["width"] = widths

    # Show statistics
    print("\n📊 Statistics (in feet or square feet):")
    for col in ["length", "width", "area_gis"]:
        series = gdf[col]
        print(f"\n▶ {col.upper()}:")
        print(f"  Mean   : {series.mean():.2f}")
        print(f"  Median : {series.median():.2f}")
        print(f"  StdDev : {series.std():.2f}")

    # Save to new shapefile
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    gdf.to_file(output_path)
    print(f"\n✅ Updated shapefile saved to: {output_path}")

# Example usage:
process_shapefile_with_bbox_dimensions(
    shapefile_path="Alabama/shapefile_output/merged_polygons.shp",
    output_path="Alabama/shapefile_output/merged_polygons_with_dimensions.shp"
)



📊 Statistics (in feet or square feet):

▶ LENGTH:
  Mean   : 462.37
  Median : 504.70
  StdDev : 107.93

▶ WIDTH:
  Mean   : 63.00
  Median : 59.52
  StdDev : 17.08

▶ AREA_GIS:
  Mean   : 26535.16
  Median : 25468.46
  StdDev : 10434.14

✅ Updated shapefile saved to: Alabama/shapefile_output/merged_polygons_with_dimensions.shp


In [9]:
# import geopandas as gpd
# import os

# def process_and_filter_by_area(shapefile_path, output_path_filtered):
#     # Load shapefile
#     gdf = gpd.read_file(shapefile_path)

#     # Ensure CRS is projected for area calculation
#     if gdf.crs.is_geographic:
#         print("🔁 Reprojecting to projected CRS for accurate area calculation...")
#         gdf = gdf.to_crs(epsg=26916)  # UTM Zone 16N

#     # Recalculate area in square meters and convert to square feet (1 m² = 10.7639 ft²)
#     gdf["area_gis"] = gdf.geometry.area * 10.7639

#     # Compute and print statistics
#     stats = gdf["area_gis"].describe()[["mean", "50%", "std"]]
#     print("📊 Area (sq ft) Statistics:")
#     print(f"  Mean   : {stats['mean']:.2f}")
#     print(f"  Median : {stats['50%']:.2f}")
#     print(f"  StdDev : {stats['std']:.2f}")

#     # Filter polygons by area range
#     gdf_filtered = gdf[(gdf["area_gis"] >= 15000) & (gdf["area_gis"] <= 37000)]

#     # Save filtered shapefile
#     os.makedirs(os.path.dirname(output_path_filtered), exist_ok=True)
#     gdf_filtered.to_file(output_path_filtered)

#     print(f"\n✅ Filtered shapefile saved to: {output_path_filtered}")
#     print(f"Remaining polygons after filtering: {len(gdf_filtered)}")


# # Example usage
# process_and_filter_by_area(
#     shapefile_path="Alabama/shapefile_output/merged_polygons.shp",
#     output_path_filtered="Alabama/shapefile_output/final_filtered_polygons.shp"
# )


📊 Area (sq ft) Statistics:
  Mean   : 26993.36
  Median : 26532.88
  StdDev : 11175.41

✅ Filtered shapefile saved to: Alabama/shapefile_output/final_filtered_polygons.shp
Remaining polygons after filtering: 3092


In [2]:
# import geopandas as gpd
# import os

# def filter_by_length_width_area(shapefile_path, output_path_filtered):
#     # Load shapefile
#     gdf = gpd.read_file(shapefile_path)

#     # Ensure CRS is projected
#     if gdf.crs.is_geographic:
#         print("🔁 Reprojecting to projected CRS for accurate measurement...")
#         gdf = gdf.to_crs(epsg=26916)  # UTM Zone 16N

#     # Ensure area_gis is in sq ft
#     if "area_gis" not in gdf.columns:
#         gdf["area_gis"] = gdf.geometry.area * 10.7639  # convert m² → ft²

#     # Apply filters
#     gdf_filtered = gdf[
#         (gdf["length"] >= 380) &
#         (gdf["width"] >= 35) & (gdf["width"] <= 100) &
#         (gdf["area_gis"] >= 15000) & (gdf["area_gis"] <= 50000)
#     ]

#     # Save filtered shapefile
#     os.makedirs(os.path.dirname(output_path_filtered), exist_ok=True)
#     gdf_filtered.to_file(output_path_filtered)

#     # Show statistics
#     print("\n📊 Statistics (Filtered Data):")
#     for col in ["length", "width", "area_gis"]:
#         print(f"\n▶ {col.upper()}:")
#         print(f"  Mean   : {gdf_filtered[col].mean():.2f}")
#         print(f"  Median : {gdf_filtered[col].median():.2f}")
#         print(f"  StdDev : {gdf_filtered[col].std():.2f}")

#     print(f"\n✅ Filtered shapefile saved to: {output_path_filtered}")
#     print(f"Remaining polygons after filtering: {len(gdf_filtered)}")


# # Example usage
# filter_by_length_width_area(
#     shapefile_path="Alabama/shapefile_output/merged_polygons_with_dimensions.shp",
#     output_path_filtered="Alabama/shapefile_output/final_filtered_by_all_criteria.shp"
# )



📊 Statistics (Filtered Data):

▶ LENGTH:
  Mean   : 498.00
  Median : 507.29
  StdDev : 71.25

▶ WIDTH:
  Mean   : 62.82
  Median : 60.07
  StdDev : 9.43

▶ AREA_GIS:
  Mean   : 28422.92
  Median : 26581.56
  StdDev : 7950.96

✅ Filtered shapefile saved to: Alabama/shapefile_output/final_filtered_by_all_criteria.shp
Remaining polygons after filtering: 10667


In [6]:
import geopandas as gpd
import os

def filter_by_length_width_area_proximity(shapefile_path, output_path_filtered):
    # Load shapefile
    gdf = gpd.read_file(shapefile_path)

    # Ensure CRS is projected
    if gdf.crs.is_geographic:
        print("🔁 Reprojecting to projected CRS for accurate measurement...")
        gdf = gdf.to_crs(epsg=26916)  # units in meters

    # Ensure area_gis is in sq ft
    if "area_gis" not in gdf.columns:
        gdf["area_gis"] = gdf.geometry.area * 10.7639

    print(f"📌 Initial shapes: {len(gdf)}")

    # Step 1: Main filters
    gdf_filtered = gdf[
        ((gdf["length"] >= 370) | (gdf["width"] >= 39) & (gdf["width"] <= 100)) &
        (gdf["area_gis"] >= 15000) & (gdf["area_gis"] <= 60000)
    ].copy()
    print(f"✅ After length/width/area filter: {len(gdf_filtered)}")

    if len(gdf_filtered) == 0:
        print("⚠️ No shapes left after main filters — check your thresholds.")
        return

    # Reset index so spatial index and dataframe align
    gdf_filtered = gdf_filtered.reset_index(drop=True)

    # Step 2: Proximity filter
    gdf_filtered["centroid"] = gdf_filtered.geometry.centroid
    radius_m = 60 

    # Build spatial index
    sindex = gdf_filtered.sindex
    keep_idxs = set()

    for idx, centroid in gdf_filtered["centroid"].items():
        # Get possible matches within bounding box
        possible_matches_idx = list(sindex.intersection(centroid.buffer(radius_m).bounds))
        # Remove self
        possible_matches_idx = [i for i in possible_matches_idx if i != idx]
        # Check actual distance
        for i in possible_matches_idx:
            if centroid.distance(gdf_filtered.loc[i, "centroid"]) <= radius_m:
                keep_idxs.add(idx)
                break

    gdf_final = gdf_filtered.loc[list(keep_idxs)].drop(columns=["centroid"])
    print(f"✅ After proximity filter: {len(gdf_final)}")

    if len(gdf_final) == 0:
        print("⚠️ No shapes left after proximity filter.")
        return

    # Step 3: Statistics
    print("\n📊 Statistics (Final Filtered Data):")
    for col in ["length", "width", "area_gis"]:
        print(f"\n▶ {col.upper()}:")
        print(f"  Mean   : {gdf_final[col].mean():.2f}")
        print(f"  Median : {gdf_final[col].median():.2f}")
        print(f"  StdDev : {gdf_final[col].std():.2f}")

    # Step 4: Save
    os.makedirs(os.path.dirname(output_path_filtered), exist_ok=True)
    gdf_final.to_file(output_path_filtered)
    print(f"\n💾 Filtered shapefile saved to: {output_path_filtered}")

# Example usage
filter_by_length_width_area_proximity(
    shapefile_path="Alabama/shapefile_output/merged_polygons_with_dimensions.shp",
    output_path_filtered="Alabama/shapefile_output/final_filtered_with_proximity.shp"
)


📌 Initial shapes: 13223
✅ After length/width/area filter: 11895
✅ After proximity filter: 11325

📊 Statistics (Final Filtered Data):

▶ LENGTH:
  Mean   : 486.60
  Median : 506.52
  StdDev : 80.88

▶ WIDTH:
  Mean   : 63.05
  Median : 59.97
  StdDev : 10.82

▶ AREA_GIS:
  Mean   : 27875.00
  Median : 26255.57
  StdDev : 8490.79

💾 Filtered shapefile saved to: Alabama/shapefile_output/final_filtered_with_proximity.shp


In [7]:
import geopandas as gpd
import pandas as pd
import os

def filter_by_length_width_area_proximity(shapefile_path, output_path_filtered):
    # Load shapefile
    gdf = gpd.read_file(shapefile_path)
    # Ensure CRS is projected
    if gdf.crs.is_geographic:
        print("🔁 Reprojecting to projected CRS for accurate measurement...")
        gdf = gdf.to_crs(epsg=26916)  # meters
    
    # Ensure area_gis in sq ft
    if "area_gis" not in gdf.columns:
        gdf["area_gis"] = gdf.geometry.area * 10.7639
    
    # Prepare class==0 mask (handles str or numeric)
    has_class = "class" in gdf.columns
    if has_class:
        class_num = pd.to_numeric(gdf["class"], errors="coerce")
        class0_mask = class_num.fillna(1) == 0
    else:
        class0_mask = pd.Series(False, index=gdf.index)
    
    print(f"📌 Initial shapes: {len(gdf)}")
    print(f"📎 class==0 present: {int(class0_mask.sum())}" if has_class else "📎 'class' field not found, no override applied")
    
    # Step 1: Main filters
    length_ok = gdf["length"] >= 370
    width_ok = (gdf["width"] >= 39) & (gdf["width"] <= 100)
    area_ok = (gdf["area_gis"] >= 15000) & (gdf["area_gis"] <= 60000)
    
    main_mask = ((length_ok) | (width_ok)) & area_ok
    
    # Apply main filters OR override for class==0
    gdf_filtered = gdf[main_mask | class0_mask].copy()
    print(f"✅ After length/width/area + class0 override: {len(gdf_filtered)}")
    
    if len(gdf_filtered) == 0:
        print("⚠️ No shapes left after filters — check thresholds.")
        return
    
    # Reset index so spatial index and dataframe align
    gdf_filtered = gdf_filtered.reset_index(drop=True)
    
    # Step 2: Proximity filter for non-class0 features only
    # We always keep class==0 features. Others must have a neighbor within radius.
    radius_m = 50
    gdf_filtered["centroid"] = gdf_filtered.geometry.centroid
    
    # Build spatial index
    sindex = gdf_filtered.sindex
    
    keep_idxs = set(gdf_filtered.index[class0_mask.reindex(gdf_filtered.index, fill_value=False)])
    
    for idx, centroid in gdf_filtered["centroid"].items():
        # Skip proximity requirement if this is class==0
        if idx in keep_idxs:
            continue
    
        # Get possible matches within bounding box
        possible = list(sindex.intersection(centroid.buffer(radius_m).bounds))
        # Remove self
        possible = [i for i in possible if i != idx]
    
        # Check actual distance
        near = False
        for i in possible:
            if centroid.distance(gdf_filtered.at[i, "centroid"]) <= radius_m:
                near = True
                break
        if near:
            keep_idxs.add(idx)
    
    gdf_final = gdf_filtered.loc[sorted(keep_idxs)].drop(columns=["centroid"])
    print(f"✅ After proximity filter + class0 override: {len(gdf_final)}")
    
    if len(gdf_final) == 0:
        print("⚠️ No shapes left after proximity filter.")
        return
    
    # Extra reporting
    kept_due_to_class0 = gdf_final.index.isin(gdf_filtered.index[class0_mask.reindex(gdf_filtered.index, fill_value=False)]).sum()
    print(f"📎 Retained due to class==0 override: {kept_due_to_class0}")
    
    # Step 3: Statistics
    print("\n📊 Statistics (Final Filtered Data):")
    for col in ["length", "width", "area_gis"]:
        if col in gdf_final.columns:
            print(f"\n▶ {col.upper()}:")
            print(f"  Mean   : {gdf_final[col].mean():.2f}")
            print(f"  Median : {gdf_final[col].median():.2f}")
            print(f"  StdDev : {gdf_final[col].std():.2f}")
    
    # Step 4: Save
    os.makedirs(os.path.dirname(output_path_filtered), exist_ok=True)
    gdf_final.to_file(output_path_filtered)
    print(f"\n💾 Filtered shapefile saved to: {output_path_filtered}")


filter_by_length_width_area_proximity(
    shapefile_path="Alabama/shapefile_output/merged_polygons_with_dimensions.shp",
    output_path_filtered="Alabama/shapefile_output/final_filtered_with_proximity_test.shp"
)


📌 Initial shapes: 13223
📎 class==0 present: 843
✅ After length/width/area + class0 override: 12028
✅ After proximity filter + class0 override: 11285
📎 Retained due to class==0 override: 728

📊 Statistics (Final Filtered Data):

▶ LENGTH:
  Mean   : 485.95
  Median : 506.44
  StdDev : 82.01

▶ WIDTH:
  Mean   : 63.23
  Median : 60.00
  StdDev : 11.57

▶ AREA_GIS:
  Mean   : 27886.80
  Median : 26266.47
  StdDev : 8622.04

💾 Filtered shapefile saved to: Alabama/shapefile_output/final_filtered_with_proximity_test.shp


In [32]:
import geopandas as gpd
import pandas as pd
import os
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
import numpy as np

def identify_poultry_farms(shapefile_path, output_path, proximity_m=50):
    # Load shapefile
    gdf = gpd.read_file(shapefile_path)

    # Ensure projected CRS for distance measurement
    if gdf.crs.is_geographic:
        print("🔁 Reprojecting to projected CRS for accurate measurements...")
        gdf = gdf.to_crs(epsg=26916)  # UTM zone (meters)

    # Get centroids of poultry houses
    gdf["centroid"] = gdf.geometry.centroid
    coords = np.array([[p.x, p.y] for p in gdf["centroid"]])

    # Apply DBSCAN clustering (eps=distance in meters, min_samples=2 to ensure at least two houses per farm)
    clustering = DBSCAN(eps=proximity_m, min_samples=2, metric="euclidean").fit(coords)
    gdf["farm_id"] = clustering.labels_

    # Remove noise points (label = -1)
    gdf_farms = gdf[gdf["farm_id"] != -1].copy()

    # Group by farm_id to compute farm center and house count
    farm_groups = gdf_farms.groupby("farm_id").agg({
        "centroid": lambda x: Point(np.mean([p.x for p in x]), np.mean([p.y for p in x])),
        "geometry": "count"  # count houses
    }).reset_index()

    farm_groups.rename(columns={"geometry": "house_count"}, inplace=True)

    # Convert to GeoDataFrame
    farm_gdf = gpd.GeoDataFrame(farm_groups, geometry="centroid", crs=gdf.crs)

    # Convert to lat/long
    farm_gdf = farm_gdf.to_crs(epsg=4326)
    farm_gdf["latitude"] = farm_gdf.geometry.y
    farm_gdf["longitude"] = farm_gdf.geometry.x

    # Save output
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    farm_gdf.to_file(output_path)

    print(f"✅ Identified {len(farm_gdf)} poultry farms.")
    print(f"💾 Saved to: {output_path}")

    return farm_gdf


# Example usage
identify_poultry_farms(
    shapefile_path="Alabama/shapefile_output/final_filtered_with_proximity_test.shp",
    output_path="Alabama/shapefile_output/poultry_farms.shp",
    proximity_m=250
)


✅ Identified 2987 poultry farms.
💾 Saved to: Alabama/shapefile_output/poultry_farms.shp


C:\Users\mza0288\AppData\Local\Temp\ipykernel_14100\312690354.py:46: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  farm_gdf.to_file(output_path)
C:\Users\mza0288\AppData\Local\anaconda3\envs\torchenv\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'house_count' to 'house_coun'
  ogr_write(


,farm_id,centroid,house_count,latitude,longitude
0,0,POINT (-85.87696 30.99901),2,30.999014,-85.876964
1,1,POINT (-85.7625 30.99126),5,30.991262,-85.762501
2,2,POINT (-85.76813 30.95426),4,30.954257,-85.768125
3,3,POINT (-85.79446 30.95071),4,30.950712,-85.794458
4,4,POINT (-85.70966 30.96255),3,30.962550,-85.709657
...,...,...,...,...,...
2982,2982,POINT (-87.3114 35.03122),2,35.031215,-87.311401
2983,2983,POINT (-87.34009 35.02348),4,35.023481,-87.340091
2984,2984,POINT (-87.33193 35.01894),2,35.018942,-87.331932
2985,2985,POINT (-87.29262 35.0117),2,35.011700,-87.292615


In [21]:

# Example usage
identify_poultry_farms(
    shapefile_path="Alabama/shapefile_output/merged_polygons_with_dimensions.shp",
    output_path="Alabama/shapefile_output/poultry_farms_test.shp",
    proximity_m=125
)


✅ Identified 3634 poultry farms.
💾 Saved to: Alabama/shapefile_output/poultry_farms_test.shp


C:\Users\mza0288\AppData\Local\Temp\ipykernel_14100\3983812881.py:46: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  farm_gdf.to_file(output_path)
C:\Users\mza0288\AppData\Local\anaconda3\envs\torchenv\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'house_count' to 'house_coun'
  ogr_write(


,farm_id,centroid,house_count,latitude,longitude
0,0,POINT (-85.87696 30.99901),2,30.999014,-85.876964
1,1,POINT (-85.7625 30.99126),5,30.991262,-85.762501
2,2,POINT (-85.76813 30.95426),4,30.954257,-85.768125
3,3,POINT (-85.79446 30.95071),4,30.950712,-85.794458
4,4,POINT (-85.70966 30.96255),3,30.962550,-85.709657
...,...,...,...,...,...
3629,3629,POINT (-87.33193 35.01894),2,35.018942,-87.331932
3630,3630,POINT (-87.29262 35.0117),2,35.011700,-87.292615
3631,3631,POINT (-88.23483 35.0533),5,35.053302,-88.234829
3632,3632,POINT (-88.24022 35.02064),3,35.020642,-88.240215
